# Digit Recognition Dataset Analysis

Living notebook tracking the composition, balance, and quality of the training dataset for the Sudoku CNN digit recognizer.

## Research Context

**Problem:** CNN trained on MNIST (handwritten) fails on printed newspaper digits (61% pipeline accuracy).

**Key finding from literature:**
- "MNIST was found to be ill-suited for recognizing computer font digits, resulting in many false positives" ([francois-rozet/sudoku](https://github.com/francois-rozet/sudoku))
- Chars74K dataset achieved 99.1% on Sudoku digit recognition ([GaneshSparkz/OCR-Sudoku-Solver](https://github.com/GaneshSparkz/OCR-Sudoku-Solver))
- SVHN backbone raised accuracy to 98.4% in one study

**Our approach:** MNIST (handwritten) + font-rendered printed digits + synthetic empty cells

**Khan et al. (2024) credibility:**
- Published in *International Journal of Multidisciplinary Research and Growth Evaluation* — low-impact journal, not top-tier
- Claims 100% accuracy on 200 images — no independent verification
- Uses MNIST + largest-contour detection — same approaches that break on real photos
- Essentially a student project writeup, not rigorous peer-reviewed research

**Open-source alternatives to explore:**
- [Chars74K](http://www.ee.surrey.ac.uk/CVSSP/demos/chars74k/) — printed + handwritten + natural characters
- [kaydee0502/printed-digits-dataset](https://github.com/kaydee0502/printed-digits-dataset) — Sudoku-specific
- [wichtounet/sudoku_dataset](https://github.com/wichtounet/sudoku_dataset) — CC-BY-4.0 Sudoku images
- [francois-rozet/sudoku](https://github.com/francois-rozet/sudoku) — 445 fonts approach

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from app.ml.dataset import create_datasets, PrintedDigitDataset, EmptyCellDataset

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Dataset Composition

In [ ]:
train_ds, val_ds, test_ds = create_datasets()
print(f'Train: {len(train_ds)}')
print(f'Val:   {len(val_ds)}')
print(f'Test:  {len(test_ds)}')

## 2. Class Balance

In [ ]:
# Count labels in training set (sample if too large)
labels = []
n = min(len(train_ds), 10000)
for i in range(0, len(train_ds), max(1, len(train_ds) // n)):
    _, label = train_ds[i]
    labels.append(label)

counts = Counter(labels)
fig, ax = plt.subplots(figsize=(10, 4))
digits = range(10)
bars = ax.bar(digits, [counts.get(d, 0) for d in digits])
ax.set_xlabel('Digit')
ax.set_ylabel('Count (sampled)')
ax.set_title('Training Set Class Distribution')
ax.set_xticks(digits)
for bar, d in zip(bars, digits):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            str(counts.get(d, 0)), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## 3. Printed Digit Samples

In [ ]:
printed = PrintedDigitDataset(count_per_digit=100, seed=42)
print(f'PrintedDigitDataset: {len(printed)} images, {len(printed.fonts)} fonts')

fig, axes = plt.subplots(3, 9, figsize=(14, 5))
fig.suptitle('Printed Digit Samples (3 per digit)', fontsize=14)
for digit in range(1, 10):
    # Find 3 examples of this digit
    examples = [i for i, l in enumerate(printed.labels) if l == digit][:3]
    for row, idx in enumerate(examples):
        img, label = printed[idx]
        axes[row, digit-1].imshow(img.squeeze(), cmap='gray')
        axes[row, digit-1].set_title(str(label), fontsize=10)
        axes[row, digit-1].axis('off')
plt.tight_layout()
plt.show()

## 4. MNIST vs Printed Comparison

In [ ]:
from torchvision import datasets, transforms

mnist = datasets.MNIST(root='../data/mnist', train=True, download=True, transform=transforms.ToTensor())

fig, axes = plt.subplots(2, 9, figsize=(14, 4))
fig.suptitle('MNIST (top) vs Printed Fonts (bottom)', fontsize=14)
for digit in range(1, 10):
    # MNIST example
    for i in range(len(mnist)):
        img, label = mnist[i]
        if label == digit:
            axes[0, digit-1].imshow(img.squeeze(), cmap='gray')
            axes[0, digit-1].set_title(f'MNIST {digit}', fontsize=9)
            axes[0, digit-1].axis('off')
            break
    # Printed example
    for i in range(len(printed)):
        if printed.labels[i] == digit:
            img, _ = printed[i]
            axes[1, digit-1].imshow(img.squeeze(), cmap='gray')
            axes[1, digit-1].set_title(f'Print {digit}', fontsize=9)
            axes[1, digit-1].axis('off')
            break
plt.tight_layout()
plt.show()

## 5. TODO / Future Analysis

- [ ] Add training curves after retraining
- [ ] Per-class accuracy comparison: MNIST-only vs MNIST+Printed
- [ ] Confusion matrix on GT images (no leakage — evaluate only)
- [ ] Evaluate Chars74K as alternative/supplement to font rendering
- [ ] Evaluate SVHN as alternative/supplement
- [ ] Check if a pretrained TF model (e.g., from Chars74K) outperforms training from scratch
- [ ] Font diversity analysis: which fonts produce which failure modes?